# Epsilon Fund — Strategy Testing
---

In [2]:
import pandas as pd
import numpy as np
import sys

# ── Set your repo root path ────────────────────────────────────────────────────
ROOT = r'/Users/justiniturregui/Desktop/epsilon/github/Epsilon-Quant-Research'  # ← change to yours
# ──────────────────────────────────────────────────────────────────────────────

sys.path.append(ROOT + '/infrastructure/data')
sys.path.append(ROOT + '/infrastructure/backtester')

from binance_client import get_binance_client, get_data, get_multiple_data
from engine import backtest

---
## Data

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `BNBUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: `365` (1y) · `730` (2y) · `1825` (5y) · `2555` (7y, recommended minimum)

In [10]:
SYMBOL   = 'BTCUSDT'
INTERVAL = '1d'
LOOKBACK = 1825

# ── Multiple pairs (optional) ──────────────────────────────────────────────────
# SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
# data_dict = get_multiple_data(client, SYMBOLS, INTERVAL, LOOKBACK)
# Access via: data_dict['BTCUSDT_1d'], data_dict['ETHUSDT_1d'] ...
# ──────────────────────────────────────────────────────────────────────────────

client = get_binance_client()
df = get_data(client, SYMBOL, INTERVAL, LOOKBACK)

print(f'{SYMBOL} | {INTERVAL} | {df.index[0].date()} -> {df.index[-1].date()} | {len(df)} rows')
df.tail(3)

BTCUSDT | 1d | 2021-03-25 -> 2026-03-23 | 1825 rows


,Open,High,Low,Close,Volume
Time,,,,,
2026-03-21,70510.73,71100.94,68571.42,68918.12,13397.16750
2026-03-22,68918.12,69588.79,67360.66,67859.00,14843.78682
2026-03-23,67859.00,71499.00,67445.18,70317.71,17670.39513


---
## Strategy

**Available columns:** `Open` `High` `Low` `Close` `Volume`

**Required output:** a `position` column — `1` long · `0` flat · `-1` short  
**Optional:** `position_size` column (0–1) to use fractional capital

> Signals are shifted 1 bar by the engine — no need to shift manually.

In [13]:
strategy_df = df.copy()

# ── Strategy logic ─────────────────────────────────────────────────────────────

# ── Optimised parameters ───────────────────────────────────────────────────────
EMA_SPAN              = 21
SWING_CAUTION         = 7
SWING_STOP            = 4
ATR_CAUTION_PERIOD    = 28
ATR_STOP_PERIOD       = 22
ATR_SIZE_PERIOD       = 9
CAUTION_THRESHOLD     = 1.855
ADX_OVERRIDE          = 56
STOP_ATR_SCALE        = 1.055
RISK_PER_TRADE        = 0.0441
MAX_LEVERAGE          = 2.8

# In-position multipliers — govern stop trailing while trade is live
STOP_MULT_POS_BOTH    = 1.5
STOP_MULT_POS_CAUTION = 0.625
STOP_MULT_POS_SHORT   = 0.346
STOP_MULT_POS_NORMAL  = 1

# Entry-day multipliers — govern initial stop on entry bar only
STOP_MULT_ENT_BOTH    = 1.388
STOP_MULT_ENT_CAUTION = 0.555
STOP_MULT_ENT_SHORT   = 0.583
STOP_MULT_ENT_NORMAL  = 1


# Indicators
strategy_df['EMA']          = strategy_df['Close'].ewm(span=EMA_SPAN, adjust=False).mean()
strategy_df['Swing_Hi_Cau'] = strategy_df['High'].rolling(window=SWING_CAUTION).max()
strategy_df['Swing_Lo_Cau'] = strategy_df['Low'].rolling(window=SWING_CAUTION).min()
strategy_df['Swing_Hi_Stp'] = strategy_df['High'].rolling(window=SWING_STOP).max()

# ATR
def calculate_atr(df, period):
    high_low    = df['High'] - df['Low']
    high_close  = np.abs(df['High'] - df['Close'].shift())
    low_close   = np.abs(df['Low']  - df['Close'].shift())
    true_range  = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return true_range.rolling(window=period).mean()

strategy_df['ATR_Cau']  = calculate_atr(strategy_df, ATR_CAUTION_PERIOD)
strategy_df['ATR_Stp']  = calculate_atr(strategy_df, ATR_STOP_PERIOD)
strategy_df['ATR_Sz']   = calculate_atr(strategy_df, ATR_SIZE_PERIOD)

# ADX
def calculate_adx(high, low, close, period=14):
    tr1 = high - low
    tr2 = abs(high - close.shift())
    tr3 = abs(low  - close.shift())
    tr  = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    up_move   = high - high.shift()
    down_move = low.shift() - low

    plus_dm  = np.where((up_move > down_move) & (up_move > 0),   up_move,   0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)

    atr      = tr.ewm(alpha=1/period, adjust=False).mean()
    plus_dm  = pd.Series(plus_dm,  index=close.index).ewm(alpha=1/period, adjust=False).mean()
    minus_dm = pd.Series(minus_dm, index=close.index).ewm(alpha=1/period, adjust=False).mean()

    plus_di  = 100 * (plus_dm  / atr)
    minus_di = 100 * (minus_dm / atr)

    dx  = 100 * abs(plus_di - minus_di) / (plus_di + minus_di)
    adx = dx.rolling(period).mean()
    return adx

strategy_df['ADX_14'] = calculate_adx(strategy_df['High'], strategy_df['Low'], strategy_df['Close'], 14)

# Caution flags
strategy_df['Caution_Long'] = (
    (strategy_df['Swing_Hi_Cau'] - strategy_df['Low']) > (CAUTION_THRESHOLD * strategy_df['ATR_Cau'])
)
strategy_df['Caution_Short'] = (
    ((strategy_df['High'] - strategy_df['Swing_Lo_Cau']) > (CAUTION_THRESHOLD * strategy_df['ATR_Cau'])) |
    (strategy_df['Close'] > strategy_df['EMA'])
)

# Entry signal
strategy_df['Entry_Long'] = (
    (strategy_df['Close'] > strategy_df['EMA']) &
    ((~strategy_df['Caution_Long']) | (strategy_df['ADX_14'] > ADX_OVERRIDE))
)

# Position sizing
strategy_df['position_size'] = (
    RISK_PER_TRADE / (strategy_df['ATR_Sz'] / strategy_df['Close'])
).clip(0.1, MAX_LEVERAGE)

# Position loop
strategy_df['position']      = 0
strategy_df['Stop_Loss']     = np.nan
strategy_df['Entry_Price']   = np.nan

in_position  = 0
stop_loss    = 0
entry_price  = 0
current_size = 0.0

warmup = max(SWING_CAUTION, ATR_CAUTION_PERIOD, ATR_STOP_PERIOD, EMA_SPAN) + 5

for i in range(warmup, len(strategy_df)):
    idx      = strategy_df.index[i]
    curr     = strategy_df.iloc[i]
    prev     = strategy_df.iloc[i - 1]

    # Stop multiplier
    # Stop multiplier
# Stop multiplier
    if in_position == 1:
        if curr['Caution_Long'] and curr['Caution_Short']:
            sm = STOP_MULT_POS_BOTH
        elif curr['Caution_Long']:
            sm = STOP_MULT_POS_CAUTION
        elif curr['Caution_Short']:
            sm = STOP_MULT_POS_SHORT
        else:
            sm = STOP_MULT_POS_NORMAL
    else:
        if prev['Entry_Long']:
            if curr['Caution_Long'] and curr['Caution_Short']:
                sm = STOP_MULT_ENT_BOTH
            elif curr['Caution_Long']:
                sm = STOP_MULT_ENT_CAUTION
            elif curr['Caution_Short']:
                sm = STOP_MULT_ENT_SHORT
            else:
                sm = STOP_MULT_ENT_NORMAL
        else:
            sm = STOP_MULT_POS_NORMAL

    stop_distance = curr['ATR_Stp'] * sm * STOP_ATR_SCALE


    # Long management
    if in_position == 1:
        if prev['Close'] < stop_loss:
            in_position  = 0
            current_size = 0.0
            strategy_df.loc[idx, ['position', 'position_size']] = [0, 0.0]
            continue
        stop_loss = max(stop_loss, curr['Swing_Hi_Stp'] - stop_distance)
        strategy_df.loc[idx, ['position', 'position_size', 'Stop_Loss']] = [
            1, current_size, stop_loss
        ]

    # Entry when flat
    else:
        if prev['Entry_Long']:
            stop_loss    = curr['Swing_Hi_Stp'] - stop_distance
            entry_price  = curr['Close']
            in_position  = 1
            current_size = curr['position_size']
            strategy_df.loc[idx, ['position', 'position_size', 'Entry_Price', 'Stop_Loss']] = [
                1, current_size, entry_price, stop_loss
            ]

# ──────────────────────────────────────────────────────────────────────────────

indicator_cols = ['EMA', 'ATR_Cau', 'ADX_14', 'Swing_Hi_Cau']
strategy_df.dropna(subset=indicator_cols, inplace=True)

strategy_df['position']      = strategy_df['position'].fillna(0).astype(int)
strategy_df['position_size'] = strategy_df['position_size'].fillna(0)
strategy_df['Stop_Loss']     = strategy_df['Stop_Loss'].fillna(0)
strategy_df['Entry_Price']   = strategy_df['Entry_Price'].fillna(0)

strategy_df['position'].value_counts()

position
0    1230
1     568
Name: count, dtype: int64

---
## Backtest

| Parameter | Options | Default |
|---|---|---|
| `cost` | Cost per trade as decimal — `0.001` = 0.1% | `0.0` |
| `show_plot` | `True` / `False` | `True` |
| `save_html` | Filename string or `None` | `None` |
| `show_trades` | Overlay entry/exit markers on price chart | `False` |
| `benchmark_data` | DataFrame with `Close` column for buy & hold comparison | same asset |

In [14]:
results = backtest(
    data           = strategy_df,
    cost           = 0.001,
    show_plot      = True,
    save_html      = None,
    show_trades    = False,
    benchmark_data = None
)

print(f"Return        {results['total_return']*100:>8.2f}%")
print(f"Sharpe        {results['sharpe_ratio']:>8.2f}")
print(f"Max Drawdown  {results['max_drawdown']*100:>8.2f}%")
print(f"Profit Factor {results['profit_factor']:>8.2f}")
print(f"Win Rate      {results['win_rate']*100:>8.2f}%")
print(f"# Trades      {results['num_trades']:>8}")

Return          712.03%
Sharpe            1.31
Max Drawdown    -49.65%
Profit Factor     1.65
Win Rate         49.12%
# Trades           283
